# HAA (Hybrid Asset Allocation) Rebalancing

Keller & Keuning (2023), 13612U 모멘텀 기반. 매월 1회 리밸런싱 전제.

이 노트북은 한국투자증권 공식 오픈API SDK(`open-trading-api/examples_user`)만 사용합니다.
`KIS_Common.py` / `KIS_API_Helper_US_ALL.py` 같은 커스텀 헬퍼에는 의존하지 않습니다.

**실행 전 확인:**
1. `~/KIS/config/kis_devlp.yaml` 에 앱키/시크릿/계좌번호가 설정되어 있어야 합니다.
2. `CONFIRM_LIVE_ORDERS = False` 로 시작하세요 — 실제 주문을 넣기 전 대시보드 형태로 리밸런싱 계획만 먼저 확인할 수 있습니다.

In [31]:
import sys, os, json, time, math
from datetime import datetime, timezone

import pandas as pd
import FinanceDataReader as fdr
import pandas_market_calendars as mcal

OPEN_API_ROOT = r"C:\projects\autotrade\KIS(hantu)\open-trading-api\examples_user"
for p in (OPEN_API_ROOT,
          os.path.join(OPEN_API_ROOT, "overseas_stock"),
          os.path.join(OPEN_API_ROOT, "domestic_stock")):
    if p not in sys.path:
        sys.path.insert(0, p)

import kis_auth as ka
from overseas_stock_functions import order, inquire_balance as us_inquire_balance, price
from domestic_stock_functions import inquire_balance as kr_inquire_balance

# ------------------------------------------------------------
# 환경 설정
# ------------------------------------------------------------
ENV = "real"                 # "real" 실전투자 / "demo" 모의투자(paper)
SVR = "prod" if ENV == "real" else "vps"
OVRS_EXCG_CD = "NASD"        # 미국 통합주문 (주문/잔고조회용)
PRICE_EXCD = "NAS"           # 현재가조회용 거래소 코드
CONFIRM_LIVE_ORDERS = False  # True 로 바꿔야 실제 매수/매도 주문이 나갑니다

ka.auth(svr=SVR) # token 발급
trenv = ka.getTREnv() # trading environment: account no, prod_no, etc
CANO = trenv.my_acct
ACNT_PRDT_CD = trenv.my_prod

BOT_NAME = f"HAA_{ENV.upper()}"
HAA_DIR = r"C:\projects\autotrade\KIS(hantu)\haa_execution"
STATE_DIR = os.path.join(HAA_DIR, "rebalancing_check")
os.makedirs(STATE_DIR, exist_ok=True)
STATE_FILE = os.path.join(STATE_DIR, f"rebalancing_check_{BOT_NAME}.json") # json file path string. the line itself is not creating the json file.

print(f"env={ENV} cano={CANO} prod={ACNT_PRDT_CD} bot={BOT_NAME}")

env=real cano=43033908 prod=01 bot=HAA_REAL


In [32]:
# ------------------------------------------------------------
# 이번 달에 이미 리밸런싱했는지 체크
# ------------------------------------------------------------
time_info = time.gmtime()
strYM = f"{time_info.tm_year}_{time_info.tm_mon}"

YMDict = {}
try:
    with open(STATE_FILE, "r") as f:
        YMDict = json.load(f)
except Exception as e:
    print("state file not found (first run):", e)

Rebalancing_Required = YMDict.get("ym_st") != strYM
print("strYM:", strYM, "Rebalancing_Required:", Rebalancing_Required)

state file not found (first run): [Errno 2] No such file or directory: 'C:\\projects\\autotrade\\KIS(hantu)\\haa_execution\\rebalancing_check\\rebalancing_check_HAA_REAL.json'
strYM: 2026_8 Rebalancing_Required: True


In [33]:
# ------------------------------------------------------------
# 미국 시장 개장 여부 (NYSE 캘린더 기준, 공휴일 반영)
# ------------------------------------------------------------
def is_us_market_open() -> bool:
    nyse = mcal.get_calendar("NYSE") #load the official NYSE trading calendar
    now_utc = datetime.now(timezone.utc) # load current datetime in UTC
    schedule = nyse.schedule(start_date=now_utc.date(), end_date=now_utc.date()) # asking if the current date is the trading date or not
    if schedule.empty: # if weekend/holiday => returning false. market closed.
        return False
    open_utc = schedule.iloc[0]["market_open"]
    close_utc = schedule.iloc[0]["market_close"]
    return open_utc <= now_utc <= close_utc # checking the range of market opening time of the trading date and see if current date time falls in between

IsMarketOpen = is_us_market_open()
print("IsMarketOpen:", IsMarketOpen)

IsMarketOpen: False


## HAA 유니버스 & 13612U 모멘텀

`momentum = (R_1m + R_3m + R_6m + R_12m) / 4`, 월말 종가 기준.


In [4]:
canary_item = "TIP"
defensive_item_list = ["BIL", "IEF"]
offensive_item_list = ["SPYM", "IWM", "EFA", "EEM", "VNQ", "PDBC", "IEF", "TLT"]

total_haa_item_list = [canary_item] + defensive_item_list + offensive_item_list
total_haa_item_list

['TIP', 'BIL', 'IEF', 'SPYM', 'IWM', 'EFA', 'EEM', 'VNQ', 'PDBC', 'IEF', 'TLT']

In [8]:
ticker='TIP'
start = (pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")
df = fdr.DataReader(ticker, start=start)

In [9]:
start

'2026-05-17'

In [10]:
df

,Open,High,Low,Close,Volume,Adj Close
2026-05-18,110.580002,110.660004,110.389999,110.480003,2490200,107.366692
2026-05-19,110.150002,110.269997,109.940002,110.110001,2623600,107.007111
2026-05-20,110.110001,110.449997,110.099998,110.370003,2915600,107.259789
2026-05-21,110.300003,110.389999,110.150002,110.370003,2792500,107.259789
2026-05-22,110.510002,110.540001,110.120003,110.379997,1410800,107.269501
...,...,...,...,...,...,...
2026-08-10,106.959999,107.000000,106.849998,106.860001,1137300,106.860001
2026-08-11,106.879997,106.989998,106.800003,106.889999,1978600,106.889999
2026-08-12,107.050003,107.070000,106.900002,106.919998,1132500,106.919998
2026-08-13,107.120003,107.309998,107.080002,107.160004,1616400,107.160004


In [11]:
df["Close"].resample("ME").last()

2026-05-31    111.209999
2026-06-30    109.430000
2026-07-31    107.629997
2026-08-31    106.989998
Freq: ME, Name: Close, dtype: float64

In [14]:
closes = get_month_end_closes(ticker)

In [15]:
closes

2025-06-30    110.040001
2025-07-31    109.669998
2025-08-31    111.169998
2025-09-30    111.220001
2025-10-31    111.379997
2025-11-30    111.260002
2025-12-31    109.910004
2026-01-31    110.480003
2026-02-28    111.879997
2026-03-31    110.360001
2026-04-30    111.559998
2026-05-31    111.209999
2026-06-30    109.430000
2026-07-31    107.629997
2026-08-31    106.989998
Freq: ME, Name: Close, dtype: float64

In [13]:
# 14 is a hard-coded number to safely get 12 month back ME close price
def get_month_end_closes(ticker: str, months_back: int = 14) -> pd.Series:
    start = (pd.Timestamp.today() - pd.DateOffset(months=months_back)).strftime("%Y-%m-%d")
    df = fdr.DataReader(ticker, start=start)
    return df["Close"].resample("ME").last().dropna()

def momentum_13612u(ticker: str) -> float:
    closes = get_month_end_closes(ticker)

    p0 = closes.iloc[-1]   # 현재(가장 최근 월말) 월말이 아니더라도 예를들어 8월14일이라 하더라도 그게 현재까지의 가장 last date이므로 이걸 월말로 침.
    p1 = closes.iloc[-2]   # 1개월 전
    p3 = closes.iloc[-4]   # 3개월 전
    p6 = closes.iloc[-7]   # 6개월 전
    p12 = closes.iloc[-13] # 12개월 전

    r1 = (p0 / p1) - 1
    r3 = (p0 / p3) - 1
    r6 = (p0 / p6) - 1
    r12 = (p0 / p12) - 1

    return (r1 + r3 + r6 + r12) / 4

In [16]:
momentum_dict = {}
for item in total_haa_item_list:
    momentum_dict[item] = momentum_13612u(item)

print("\n=== 모멘텀 스코어 ===")
for k, v in momentum_dict.items():
    print(f"{k:>6}: {v:+.4f}")


=== 모멘텀 스코어 ===
   TIP: -0.0313
   BIL: -0.0017
   IEF: -0.0247
  SPYM: +0.1004
   IWM: +0.1406
   EFA: +0.0710
   EEM: +0.1027
   VNQ: +0.0339
  PDBC: +0.1471
   TLT: -0.0488


## 캐너리 체크 → 공격/방어 자산 배분 결정

In [17]:
def pick_best_defensive() -> str:
    bil_momentum = momentum_dict.get("BIL")
    ief_momentum = momentum_dict.get("IEF")
    if bil_momentum is None or ief_momentum is None:
        raise ValueError("BIL, IEF 모멘텀 둘 다 계산 실패 - 진행 불가")
    return "BIL" if bil_momentum >= ief_momentum else "IEF"


target_portfolio = {}
tip_momentum = momentum_dict.get(canary_item)
if tip_momentum is None:
    raise ValueError("캐너리(TIP) 모멘텀 계산 실패 - 진행 불가")

if tip_momentum <= 0:
    best_defensive = pick_best_defensive()
    target_portfolio[best_defensive] = 1.0
    print(f"[캐너리 OFF] TIP 모멘텀 {tip_momentum:.4f} <= 0 -> {best_defensive} 100%")
else:
    momentum_df = pd.DataFrame(list(momentum_dict.items()), columns=["stock_code", "momentum"])
    momentum_df.sort_values(by="momentum", ascending=False, inplace=True)
    top4 = momentum_df.iloc[:4]

    for ticker in top4["stock_code"]:
        if momentum_dict.get(ticker) > 0:
            target_portfolio[ticker] = 0.25
        else:
            target_portfolio[pick_best_defensive()] = target_portfolio.get(pick_best_defensive(), 0) + 0.25

total_weight = sum(target_portfolio.values())
assert abs(total_weight - 1.0) < 1e-6, f"비중 합이 100%가 아닙니다: {total_weight}"
target_portfolio

[캐너리 OFF] TIP 모멘텀 -0.0313 <= 0 -> BIL 100%


{'BIL': 1.0}

## 계좌 잔고 조회 (공식 SDK)

- KRW 예수금(투자 가능 금액): 국내주식 잔고조회 `inquire_balance()` output2의 `dnca_tot_amt`
- 현재 보유 중인 미국 종목: 해외주식 잔고조회 `inquire_balance()` output1
- 한국 주식은 절대 건드리지 않습니다. KRW 예수금만 투자용 금액입니다.

In [18]:
import requests
from bs4 import BeautifulSoup

def get_usdkrw_from_naver() -> float:
    url = "https://finance.naver.com/marketindex/exchangeList.naver"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    res = requests.get(url, headers=headers, timeout=5)
    res.raise_for_status()
    res.encoding = "euc-kr"
    soup = BeautifulSoup(res.text, "html.parser")
    table = soup.find("table", {"class": "tbl_exchange"}) or soup.find("table")
    for row in table.find_all("tr"):
        row_text = row.get_text(" ", strip=True)
        if "USD" not in row_text:
            continue
        for col in row.find_all("td"):
            text = col.get_text(strip=True).replace(",", "")
            try:
                value = float(text)
                if 500 < value < 3000:
                    return value
            except ValueError:
                continue
    raise RuntimeError("USD 환율을 찾지 못함 - 페이지 구조 확인 필요")

exchange_rate = get_usdkrw_from_naver()
print("USD/KRW:", exchange_rate)

USD/KRW: 1415.2


In [20]:
df1, _ = us_inquire_balance(
        cano=CANO, acnt_prdt_cd=ACNT_PRDT_CD,
        ovrs_excg_cd=OVRS_EXCG_CD, tr_crcy_cd="USD", env_dv=ENV,
    )

INFO - Data fetch complete.


In [23]:
_, df2 = kr_inquire_balance(
    env_dv=ENV, cano=CANO, acnt_prdt_cd=ACNT_PRDT_CD,
    afhr_flpr_yn="N", inqr_dvsn="02", unpr_dvsn="01",
    fund_sttl_icld_yn="N", fncg_amt_auto_rdpt_yn="N", prcs_dvsn="01",
)

INFO - Data fetch complete.


In [24]:
df2

,dnca_tot_amt,nxdy_excc_amt,prvs_rcdl_excc_amt,cma_evlu_amt,bfdy_buy_amt,thdt_buy_amt,nxdy_auto_rdpt_amt,bfdy_sll_amt,thdt_sll_amt,d2_auto_rdpt_amt,...,tot_evlu_amt,nass_amt,fncg_gld_auto_rdpt_yn,pchs_amt_smtl_amt,evlu_amt_smtl_amt,evlu_pfls_smtl_amt,tot_stln_slng_chgs,bfdy_tot_asst_evlu_amt,asst_icdc_amt,asst_icdc_erng_rt
0,2310328,2310328,2310328,0,0,0,0,0,0,0,...,3955328,3955328,,2555000,1645000,-910000,0,1903328,2052000,0.00000000


In [ ]:
# KRW 현금예수금 조회 (	dnca_tot_amt)
def get_krw_cash() -> float:
    _, df2 = kr_inquire_balance(
        env_dv=ENV, cano=CANO, acnt_prdt_cd=ACNT_PRDT_CD,
        afhr_flpr_yn="N", inqr_dvsn="02", unpr_dvsn="01",
        fund_sttl_icld_yn="N", fncg_amt_auto_rdpt_yn="N", prcs_dvsn="01",
    )
    if df2.empty:
        return 0.0
    return float(df2["dnca_tot_amt"].iloc[0])

# 
def get_us_holdings() -> pd.DataFrame:
    df1, _ = us_inquire_balance(
        cano=CANO, acnt_prdt_cd=ACNT_PRDT_CD,
        ovrs_excg_cd=OVRS_EXCG_CD, tr_crcy_cd="USD", env_dv=ENV,
    )
    if df1.empty:
        return pd.DataFrame(columns=["stock_code", "qty_current"])
    out = pd.DataFrame({
        "stock_code": df1["ovrs_pdno"],
        "qty_current": df1["ovrs_cblc_qty"].astype(float),
    })
    return out[out["qty_current"] > 0].reset_index(drop=True)


krw_cash = get_krw_cash()
us_holdings_df = get_us_holdings()
print("KRW cash:", format(round(krw_cash), ","))
us_holdings_df

INFO - Data fetch complete.
INFO - Data fetch complete.


KRW cash: 2,310,328


,stock_code,qty_current


In [25]:
invest_rate = 1.0
total_execution_amount = krw_cash * invest_rate
print("HAA 전략에 투입될 총 금액:", format(round(total_execution_amount), ","))

target_df = pd.DataFrame([
    {"stock_code": code, 
     "stock_target_rate": weight * 100, 
     "stock_rebalance_amt": total_execution_amount * weight}
    for code, weight in target_portfolio.items()
])

HAA 전략에 투입될 총 금액: 2,310,328


In [26]:
target_df

,stock_code,stock_target_rate,stock_rebalance_amt
0,BIL,100.0,2310328.0


## 목표 포트폴리오 → 리밸런싱 계획

In [28]:
invest_rate = 1.0
total_execution_amount = krw_cash * invest_rate
print("HAA 전략에 투입될 총 금액:", format(round(total_execution_amount), ","))

target_df = pd.DataFrame([
    {"stock_code": code, 
     "stock_target_rate": weight * 100, 
     "stock_rebalance_amt": total_execution_amount * weight}
    for code, weight in target_portfolio.items()
])

def get_current_price_usd(ticker: str) -> float:
    # 종목별로 실제 상장 거래소가 다르므로(NASDAQ/NYSE Arca/AMEX 등)
    # 현재가조회는 order()의 "NASD"(미국통합) 같은 단일 코드를 지원하지 않습니다.
    # 조회가 될 때까지 순서대로 시도합니다.
    for excd in ("NAS", "NYS", "AMS"):
        df = price(auth="", excd=excd, symb=ticker)
        if df.empty or "last" not in df.columns:
            continue
        last = df["last"].iloc[0]
        if last not in (None, ""):
            return float(last)
    raise RuntimeError(f"{ticker}: 현재가 조회 실패 (NAS/NYS/AMS 모두 실패)")

target_df["current_price_usd"] = target_df["stock_code"].apply(get_current_price_usd)
target_df["current_price_krw"] = target_df["current_price_usd"] * exchange_rate
target_df["qty_target"] = (target_df["stock_rebalance_amt"] / target_df["current_price_krw"]).apply(math.trunc)
target_df

HAA 전략에 투입될 총 금액: 2,310,328


INFO - Data fetch complete.
INFO - Data fetch complete.
INFO - Data fetch complete.


,stock_code,stock_target_rate,stock_rebalance_amt,current_price_usd,current_price_krw,qty_target
0,BIL,100.0,2310328.0,91.5415,129549.5308,17


In [29]:
rebalance_df = target_df.merge(us_holdings_df, on="stock_code", how="outer").fillna(0)
rebalance_df["gap_qty"] = rebalance_df["qty_target"] - rebalance_df["qty_current"]
rebalance_df["sell_buy"] = ""
rebalance_df.loc[rebalance_df["gap_qty"] > 0, "sell_buy"] = "buy"
rebalance_df.loc[rebalance_df["gap_qty"] < 0, "sell_buy"] = "sell"
rebalance_df

C:\Users\whyyo\AppData\Local\Temp\ipykernel_15276\867426770.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rebalance_df = target_df.merge(us_holdings_df, on="stock_code", how="outer").fillna(0)


,stock_code,stock_target_rate,stock_rebalance_amt,current_price_usd,current_price_krw,qty_target,qty_current,gap_qty,sell_buy
0,BIL,100.0,2310328.0,91.5415,129549.5308,17,0,17,buy


## 리밸런싱 실행

시장이 열려 있고 이번 달에 아직 리밸런싱하지 않았을 때만 실행됩니다.
`CONFIRM_LIVE_ORDERS`가 `True`일 때만 실제로 `order()`가 호출됩니다 — 그 전까지는
계획만 출력하고 아무 주문도 나가지 않습니다.

In [30]:
if Rebalancing_Required and IsMarketOpen:
    # sell first
    sell_df = rebalance_df.loc[rebalance_df['sell_buy']== 'sell']
    
    for _, row in sell_df:
        stock_code = row["stock_code"]
        gap_qty = row["gap_qty"]
        side = row["sell_buy"] # 여기서는 당연히 sell.
        qty = gap_qty
        
        if CONFIRM_LIVE_ORDERS:
            df = order(
                cano = CANO,
                acnt_prdt_cd = ACNT_PRDT_CD,
                ovrs_excg_cd = OVRS_EXCG_CD,
                pdno = stock_code,
                ord_qty = str(qty),
                ovrs_ord_unpr = "0",
                ord_dv = side,
                ctac_tlno="",
                mgco_aptm_odno="",
                ord_svr_dvsn_cd="0",
                ord_dvsn="00", # 시장가주문.
                env_dv = ENV
            )
        
    # buy after selling
    buy_df = rebalance_df.loc[rebalance_df['sell_buy']== 'buy']
    
    for _, row in sell_df:
        stock_code = row["stock_code"]
        gap_qty = row["gap_qty"]
        side = row["sell_buy"] # 여기서는 당연히 buy.
        qty = gap_qty
        
        if CONFIRM_LIVE_ORDERS:
            df = order(
                cano = CANO,
                acnt_prdt_cd = ACNT_PRDT_CD,
                ovrs_excg_cd = OVRS_EXCG_CD,
                pdno = stock_code,
                ord_qty = str(qty),
                ovrs_ord_unpr = "0",
                ord_dv = side,
                ctac_tlno="",
                mgco_aptm_odno="",
                ord_svr_dvsn_cd="0",
                ord_dvsn="00", # 시장가주문.
                env_dv = ENV
            )
    # rebalancing flag saving
    if CONFIRM_LIVE_ORDERS:
        YMDict["ym_st"] = strYM
        with open(STATE_FILE, "w") as f: # json file 생성.
            json.dump(YMDict, f) # json file 에 리밸런싱한 year month 입력
        print("\n리밸런싱 완료 및 기록 저장됨.")
    else:
        print("\nCONFIRM_LIVE_ORDERS=False - 주문 미실행, 계획만 출력했습니다.")

In [ ]:
if Rebalancing_Required and IsMarketOpen:
    for _, row in rebalance_df[rebalance_df["gap_qty"] != 0].iterrows(): # only run for loop when the gap_quantity not equals zero.
        stock_code = row["stock_code"]
        gap_qty = int(row["gap_qty"])
        side = row["sell_buy"]
        qty = abs(gap_qty)

        print(f"[{side.upper()}] {stock_code} x {qty}")

        if CONFIRM_LIVE_ORDERS:
            df = order(
                cano=CANO,
                acnt_prdt_cd=ACNT_PRDT_CD,
                ovrs_excg_cd=OVRS_EXCG_CD, # 해외거래소코드.  NASD 로 KIS에서 지정되어 있음.
                pdno=stock_code,
                ord_qty=str(qty), # 주문수량을 interger 가 아닌 string 으로 보내게 되어 있음. API에.
                ovrs_ord_unpr="0",       # 해외주문단가를 0으로 하면 시장가로 하겠다는 뜻
                ord_dv=side, # 주문구분: sell/buy 중 1개
                ctac_tlno="", # 연락전화번호.. 이거는 그냥 빈칸으로..
                mgco_aptm_odno="", # 운용사지정번호.. 이거도 그냥 빈칸으로..
                ord_svr_dvsn_cd="0", # 주문서버구분코드, 통상적으로 0이 기본 값
                ord_dvsn="00", # 시장가주문.
                env_dv=ENV,
            )
            print(df)
            time.sleep(0.5)

    if CONFIRM_LIVE_ORDERS:
        YMDict["ym_st"] = strYM
        with open(STATE_FILE, "w") as f:
            json.dump(YMDict, f)
        print("\n리밸런싱 완료 및 기록 저장됨.")
    else:
        print("\nCONFIRM_LIVE_ORDERS=False - 주문 미실행, 계획만 출력했습니다.")
else:
    if not Rebalancing_Required:
        print("이번 달 리밸런싱은 이미 완료되었습니다.")
    if not IsMarketOpen:
        print("현재 시장이 열려있지 않습니다.")